# P6 / 01: data, PCA residuals and rolling network fits

Protocol: [`001_claude_p6_deep_stat_arb_preregistration.md`](../../discussions/Deep_Statistical_Arbitrage/001_claude_p6_deep_stat_arb_preregistration.md).

**Data.** Yahoo daily 2005–2015 is spliced with Alpaca daily 2016–2025 on returns.

**Residuals and inputs.** PCA residuals with 5 factors on a 252-session window, re-estimated daily. The inputs are the 30-session cumulative residuals.

**Fits.** Rolling: train on 750 sessions and validate on the next 250, then predict the following 125 sessions. Each fit is an ensemble of 5 seeds. The first prediction is on 2017-01-03.

Every fit is cached, so the notebook can be stopped and restarted.

In [1]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p6, s04, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years, session_schedule
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P6_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p6.P6Config()
CACHE = work_dir() / "results" / ("p6_synthetic" if SYNTHETIC else "p6")
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, train_days=300, valid_days=100, refit_every=125, first_test="2017-01-03", end="2017-12-29",
                  seeds=(0, 1), epochs=3, patience=3, n_placebo=50, n_boot=2000)
import torch
from equity_trading import p6_model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " -- CUDA not available, training on CPU (slower)"))
print(f"synthetic={SYNTHETIC} | fits every {CFG.refit_every} sessions | seeds {CFG.seeds}")

torch 2.14.0+cu126 | device cuda (NVIDIA GeForce GTX 1070)
synthetic=False | fits every 125 sessions | seeds (0, 1, 2, 3, 4)


## 1. Daily returns (Yahoo before 2016-01-04, Alpaca after), measured costs

In [2]:
t0 = time.time()
if SYNTHETIC:
    R = p6.synthetic_returns(n_days=1800, n_stocks=80, rev=0.3, start="2011-01-03")
    PRICE, COST = None, pd.DataFrame(2e-4, index=R.index, columns=R.columns)
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"])
    sessions = pd.DatetimeIndex(pd.to_datetime(list(session_schedule(list(range(2005, 2026))).index)))
    YH = p6.load_yahoo_daily(STOCKS, CFG.yahoo_start, "2016-01-06")
    download_years(STOCKS, "1Day", list(range(2016, 2026)), workers=8)
    AC, _, _ = p1.load_daily(STOCKS, list(range(2016, 2026)))
    R = p6.splice_returns(YH, AC, sessions, CFG.splice)
    R = R.loc[:, R.notna().sum() > 300]
    PRICE = AC.reindex(columns=R.columns)
    qc = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
    sess = s05b.sample_sessions([2017, 2018, 2019], 4, seed=42) + s05b.sample_sessions(list(range(2020, 2026)), 4, seed=42)
    samples = pd.concat([pd.read_parquet(qc / f"{d}.parquet") for d in sess], ignore_index=True)
    COST = p1.daily_cost(R.index, list(R.columns), s05b.cost_model(samples))   # before 2017: symbol medians
cov = R.notna().groupby(R.index.year).mean().mean(axis=1)
print(f"{R.shape[1]} stocks, {len(R)} sessions {R.index[0].date()} to {R.index[-1].date()}; ready in {time.time() - t0:.0f}s")
display(cov.rename("share of stock-days with a return").to_frame().T.round(2))

Yahoo daily: 0/503 cached; downloading 503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
D

  50/503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
D:\Anaconda\envs\equity-trading\Lib\site-p

  100/503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\A

... (14 outputs left out)

  500/503
  503/503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
D

1Day: 0 batches to download (0 symbol-years), 8 parallel requests


ValueError: Array conditional must be same shape as self

## 2. PCA residuals and inputs (cached)

In [ ]:
t0 = time.time()
ds_path = CACHE / f"dataset_p6{p6.P6_VERSION}.npz"
SCHED = p6.fit_schedule(R.index, CFG)
need_from = max(0, SCHED[0]["train"][0] - CFG.lookback)
if ds_path.exists():
    DS = p6.load_dataset(ds_path)
else:
    DS = p6.build_dataset(R, PRICE, COST, CFG, start_row=need_from)
    p6.save_dataset(DS, ds_path)
print(f"dataset {DS['x'].shape}, usable stock-days from the first training row: {DS['ok'][need_from:].mean():.1%}; {time.time() - t0:.0f}s")
print(f"{len(SCHED)} fits; first trains on {R.index[SCHED[0]['train'][0]].date()} - {R.index[SCHED[0]['valid'][1] - 1].date()}, "
      f"predicts from {SCHED[0]['first_day']}")

## 3. Rolling fits

In [ ]:
SC = np.full(DS["ok"].shape, np.nan, np.float32)
INFO, t_all = [], time.time()
for j, f in enumerate(SCHED):
    tr, va, pr = (np.arange(*f[k]) for k in ("train", "valid", "predict"))
    tr, va = tr[DS["ok"][tr].sum(1) > 0], va[DS["ok"][va].sum(1) > 0]
    seed_scores = []
    for s in CFG.seeds:
        p = CACHE / f"fit{j:02d}_seed{s}.npz"
        if p.exists():
            z = np.load(p, allow_pickle=True)
            seed_scores.append(z["scores"]); info = json.loads(str(z["info"]))
        else:
            t1 = time.time()
            model, info = p6_model.fit(DS, tr, va, seed=s, epochs=CFG.epochs, patience=CFG.patience, lr=CFG.lr,
                                       chunk=CFG.chunk_days, device=DEVICE)
            sc = p6_model.predict_scores(model, DS, pr, chunk=CFG.chunk_days)
            info["seconds"] = time.time() - t1
            np.savez(p, scores=sc, info=json.dumps(info))
            seed_scores.append(sc)
        INFO.append({"fit": j, "seed": s, "first_day": f["first_day"], **info})
    # ensemble: mean of seed scores after standardising each seed's scores per day
    zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
    SC[pr] = np.nanmean(np.stack(zs), 0)
    done = j + 1
    el = time.time() - t_all
    print(f"fit {done}/{len(SCHED)} ({f['first_day']}): valid Sharpe by seed "
          f"{[round(i['best_valid_sharpe'], 2) for i in INFO[-len(CFG.seeds):]]}; {el / 60:.1f} min, "
          f"~{el / done * (len(SCHED) - done) / 60:.0f} min left", flush=True)
np.savez(CACHE / "scores.npz", scores=SC)
INFO = pd.DataFrame(INFO)
OUT = ROOT / "notebooks" / "Deep_Statistical_Arbitrage" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
INFO.to_csv(OUT / "fits.csv", index=False, float_format="%.5g")
display(INFO.groupby("fit")[["best_valid_sharpe", "train_sharpe", "epochs"]].mean().round(3))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE),
                                               "device": DEVICE, "torch": torch.__version__, "p6": p6.P6_VERSION}, indent=2))